In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("TelcoCustomerChurn.csv")

In [9]:
df.head()

,Gender,Age,SeniorCitizen,Married,Dependents,NumberofDependents,ReferredaFriend,Number_of_Referrals,TenureinMonths,Offer,...,PaperlessBilling,PaymentMethod,MonthlyCharge,TotalCharges,TotalRefunds,TotalExtraDataCharges,TotalLongDistanceCharges,TotalRevenue,SatisfactionScore,ChurnLabel
0,Male,78,Yes,No,No,0,No,0,1,NaN,...,Yes,Bank Withdrawal,39.65,39.65,0.00,20,0.00,59.65,3,Yes
1,Female,74,Yes,Yes,Yes,1,Yes,1,8,Offer E,...,Yes,Credit Card,80.65,633.30,0.00,0,390.80,1024.10,3,Yes
2,Male,71,Yes,No,Yes,3,No,0,18,Offer D,...,Yes,Bank Withdrawal,95.45,1752.55,45.61,0,203.94,1910.88,2,Yes
3,Female,78,Yes,Yes,Yes,1,Yes,1,25,Offer C,...,Yes,Bank Withdrawal,98.50,2514.50,13.43,0,494.00,2995.07,2,Yes
4,Female,80,Yes,Yes,Yes,1,Yes,1,37,Offer C,...,Yes,Bank Withdrawal,76.50,2868.15,0.00,0,234.21,3102.36,2,Yes


In [4]:
df.columns

Index(['CustomerID', 'Gender', 'Age', 'Under30', 'SeniorCitizen', 'Married',
       'Dependents', 'NumberofDependents', 'Country', 'State', 'City',
       'ZipCode', 'Latitude', 'Longitude', 'Population', 'Quarter',
       'ReferredaFriend', 'Number_of_Referrals', 'TenureinMonths', 'Offer',
       'PhoneService', 'AvgMonthlyLongDistanceCharges', 'MultipleLines',
       'InternetService', 'InternetType', 'AvgMonthlyGBDownload',
       'OnlineSecurity', 'OnlineBackup', 'DeviceProtectionPlan',
       'PremiumTechSupport', 'StreamingTV', 'StreamingMovies',
       'StreamingMusic', 'UnlimitedData', 'Contract', 'PaperlessBilling',
       'PaymentMethod', 'MonthlyCharge', 'TotalCharges', 'TotalRefunds',
       'TotalExtraDataCharges', 'TotalLongDistanceCharges', 'TotalRevenue',
       'SatisfactionScore', 'CustomerStatus', 'ChurnLabel', 'ChurnScore',
       'CLTV', 'ChurnCategory', 'ChurnReason'],
      dtype='object')

In [5]:
drop_cols = [
    'CustomerID',
    'Country', 'State', 'City', 'ZipCode',
    'Latitude', 'Longitude',
    'CustomerStatus', 'ChurnScore', 'CLTV',
    'ChurnCategory', 'ChurnReason',
    'Under30','Population', 'Quarter'
]

In [7]:
df=df.drop(drop_cols,axis=1)

In [8]:
df.columns

Index(['Gender', 'Age', 'SeniorCitizen', 'Married', 'Dependents',
       'NumberofDependents', 'ReferredaFriend', 'Number_of_Referrals',
       'TenureinMonths', 'Offer', 'PhoneService',
       'AvgMonthlyLongDistanceCharges', 'MultipleLines', 'InternetService',
       'InternetType', 'AvgMonthlyGBDownload', 'OnlineSecurity',
       'OnlineBackup', 'DeviceProtectionPlan', 'PremiumTechSupport',
       'StreamingTV', 'StreamingMovies', 'StreamingMusic', 'UnlimitedData',
       'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharge',
       'TotalCharges', 'TotalRefunds', 'TotalExtraDataCharges',
       'TotalLongDistanceCharges', 'TotalRevenue', 'SatisfactionScore',
       'ChurnLabel'],
      dtype='object')

In [13]:
df.isnull().sum()

Gender                           0
Age                              0
SeniorCitizen                    0
Married                          0
Dependents                       0
NumberofDependents               0
ReferredaFriend                  0
Number_of_Referrals              0
TenureinMonths                   0
Offer                            0
PhoneService                     0
AvgMonthlyLongDistanceCharges    0
MultipleLines                    0
InternetService                  0
InternetType                     0
AvgMonthlyGBDownload             0
OnlineSecurity                   0
OnlineBackup                     0
DeviceProtectionPlan             0
PremiumTechSupport               0
StreamingTV                      0
StreamingMovies                  0
StreamingMusic                   0
UnlimitedData                    0
Contract                         0
PaperlessBilling                 0
PaymentMethod                    0
MonthlyCharge                    0
TotalCharges        

In [11]:
df['Offer'].unique()

array([nan, 'Offer E', 'Offer D', 'Offer C', 'Offer B', 'Offer A'],
      dtype=object)

In [12]:
df["InternetType"] = df["InternetType"].fillna("No Internet")
df["Offer"] = df["Offer"].fillna("No Offer")

"""
Missing values in categorical features such as Offer and InternetType were treated as separate categories,
as they likely represent absence of service or promotion rather than random missingness.
"""

In [29]:
X = df.drop("ChurnLabel", axis=1)
X = X.drop(columns=["SatisfactionScore"],axis=1)
y = df["ChurnLabel"]

In [30]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [31]:
categorical_cols = X_train.select_dtypes(include="object").columns
numeric_cols = X_train.select_dtypes(exclude="object").columns

In [ ]:
# Convert categorical variables into numerical format using one-hot encoding
# drop_first=True helps avoid dummy variable trap (multicollinearity)
X_train = pd.get_dummies(X_train, drop_first=True)
X_test = pd.get_dummies(X_test, drop_first=True)

# Align X_test columns with X_train columns
# This ensures both datasets have the same features after encoding
# Any missing columns in X_test will be filled with 0
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

In [ ]:
# Import StandardScaler for feature scaling
# Standardization transforms features to have mean = 0 and standard deviation = 1
from sklearn.preprocessing import StandardScaler

# Create scaler object
scaler = StandardScaler()

# Fit the scaler on training data and transform it
# We fit only on training data to prevent data leakage
X_train_scaled = scaler.fit_transform(X_train)

# Use the same fitted scaler to transform test data
# Important: Do NOT fit on test data
X_test_scaled = scaler.transform(X_test)

In [34]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)

model.fit(X_train_scaled, y_train)

LogisticRegression(max_iter=1000)

In [35]:
y_pred = model.predict(X_test_scaled)
y_prob = model.predict_proba(X_test_scaled)[:, 1]

In [36]:
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

[[953  82]
 [121 253]]
              precision    recall  f1-score   support

          No       0.89      0.92      0.90      1035
         Yes       0.76      0.68      0.71       374

    accuracy                           0.86      1409
   macro avg       0.82      0.80      0.81      1409
weighted avg       0.85      0.86      0.85      1409

ROC-AUC: 0.9095119997933298


In [ ]:
# Create a DataFrame to display feature names with their corresponding model coefficients
coef_df = pd.DataFrame({
    "Feature": X_train.columns,        # Feature names
    "Coefficient": model.coef_[0]      # Coefficients learned by the model
})

# Sort features by absolute coefficient value (importance)
# key=abs ensures we sort based on magnitude, not direction
# ascending=False gives highest impact features first
coef_df = coef_df.sort_values(by="Coefficient", key=abs, ascending=False)

# Display top 10 most influential features
print(coef_df.head(10))

                     Feature  Coefficient
6              MonthlyCharge     2.040271
2        Number_of_Referrals    -1.690958
37         Contract_Two Year    -1.244997
3             TenureinMonths    -0.906986
15            Dependents_Yes    -0.632625
36         Contract_One Year    -0.601572
26  InternetType_Fiber Optic    -0.600584
16       ReferredaFriend_Yes     0.596305
22          PhoneService_Yes    -0.528729
33       StreamingMovies_Yes    -0.411950


# Random Forest Classifier

In [ ]:
# Import Random Forest classifier
from sklearn.ensemble import RandomForestClassifier

# Initialize the Random Forest model
# n_estimators=300 → number of trees in the forest
# max_depth=10 → maximum depth of each tree (controls overfitting)
# min_samples_split=5 → minimum samples required to split a node
# random_state=42 → ensures reproducibility
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    min_samples_split=5,
    random_state=42
)

# Train the model on training data
rf.fit(X_train, y_train)

# Predict class labels on test data
y_pred_rf = rf.predict(X_test)

# Predict probability of the positive class (class 1)
# [:, 1] selects probability for churn (assuming 1 = churn)
y_prob_rf = rf.predict_proba(X_test)[:, 1]

In [43]:
from sklearn.metrics import classification_report, roc_auc_score

print(classification_report(y_test, y_pred_rf))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_rf))

              precision    recall  f1-score   support

          No       0.86      0.93      0.90      1035
         Yes       0.76      0.59      0.67       374

    accuracy                           0.84      1409
   macro avg       0.81      0.76      0.78      1409
weighted avg       0.84      0.84      0.84      1409

ROC-AUC: 0.8994497403704564


In [ ]:
import pandas as pd

# Create a DataFrame to map each feature with its importance score
feature_importance = pd.DataFrame({
    "Feature": X_train.columns,              # Feature names
    "Importance": rf.feature_importances_    # Importance score from Random Forest
})

# Sort features by importance in descending order
# Highest contributing features appear first
feature_importance = feature_importance.sort_values(by="Importance", ascending=False)

# Display top 10 most important features
print(feature_importance.head(10))

                     Feature  Importance
3             TenureinMonths    0.096321
2        Number_of_Referrals    0.094600
37         Contract_Two Year    0.087620
6              MonthlyCharge    0.073922
11              TotalRevenue    0.071832
7               TotalCharges    0.065322
26  InternetType_Fiber Optic    0.051074
10  TotalLongDistanceCharges    0.050229
36         Contract_One Year    0.040433
5       AvgMonthlyGBDownload    0.039601


# The dataset exhibited strong linear separability driven by features such as tenure, contract type, and monthly charges. As a result, Logistic Regression performed comparably to Random Forest, indicating limited nonlinear complexity in the data.